# Configurable $a_1$--$a_2$ prior and post-fit contours

This notebook reproduces the coefficient-contour comparison from `postfit_physical_parameters.ipynb` without running the full plotting suite. Edit the configuration cell to choose the priors and post-fit distributions to display. Dashed contours are priors; solid contours are post-fit distributions.

In [ ]:
from pathlib import Path
import sys

repo = Path.cwd().resolve()
while repo.name != "axial_mass" and repo != repo.parent:
    repo = repo.parent
if repo.name != "axial_mass":
    raise RuntimeError("Run this notebook from within the axial_mass repository")
helper_dir = repo / "ma_zexp" / "python" / "scripts"
if str(helper_dir) not in sys.path:
    sys.path.insert(0, str(helper_dir))

from postfit_physical_parameters import (
    FIGURE_ROOT, REFERENCE_PRIORS, SPECS, load_fit,
    plot_distribution_overlay,
)

## Configuration

`PRIORS` and `POSTFITS` are independent, so either list may be empty. Standalone prior keys are `deuterium`, `deuterium_k6`, `minerva_k6`, `lqcd_k6`, and `minerva_lqcd_k6`; any z-expansion fit key listed by the next cell can also supply its fitted prior. Post-fit keys must exist in the selected suite. Use `deuterium_k6`, rather than native-basis `deuterium`, when comparing with the other $k_{\max}=6$ distributions.

In [ ]:
SUITE = "nuwro_fit_results"

# Dashed contours.
PRIORS = [
    "deuterium_k6",
    "minerva_k6",
    "lqcd_k6",
    "minerva_lqcd_k6"
]

# Solid contours. These are loaded from the MCMC output for SUITE.
POSTFITS = [
    "minerva_k6_uniform",
]

BURN_IN = 0
THIN = 1
N_PRIOR_SAMPLES = 100_000
BINS = 55
FIGSIZE = (7.0, 6.2)
SAVE_FIGURE = True
OUTPUT_STEM = "a1_a2_prior_postfit_contours"
SAVE_DPI = 600

In [ ]:
standalone_prior_keys = (
    "deuterium", "deuterium_k6", *REFERENCE_PRIORS.keys(),
)
fit_specs = {spec.key: spec for spec in SPECS if spec.prior is not None}
print("Standalone priors:", ", ".join(standalone_prior_keys))
print("Z-expansion fit keys:", ", ".join(fit_specs))

## Load selected fit outputs

A selected fit key is loaded once even if both its prior and post-fit distribution are requested. Standalone priors are sampled directly and do not require fit output.

In [ ]:
unknown_postfits = sorted(set(POSTFITS) - set(fit_specs))
unknown_priors = sorted(
    set(PRIORS) - set(fit_specs) - set(standalone_prior_keys)
)
if unknown_postfits:
    raise KeyError(f"Unknown post-fit key(s): {unknown_postfits}")
if unknown_priors:
    raise KeyError(f"Unknown prior key(s): {unknown_priors}")
if not PRIORS and not POSTFITS:
    raise ValueError("Select at least one prior or post-fit distribution")

# Fit priors need loading only when they do not also exist as standalone references.
fit_prior_keys = set(PRIORS) - set(standalone_prior_keys)
keys_to_load = sorted(set(POSTFITS) | fit_prior_keys)
results = {}
for key in keys_to_load:
    result = load_fit(
        fit_specs[key], SUITE, burn_in=BURN_IN, thin=THIN,
        n_prior=N_PRIOR_SAMPLES,
    )
    if result is None:
        raise FileNotFoundError(
            f"No unique PROfile ROOT file found for {key!r} in suite {SUITE!r}"
        )
    results[key] = result
    print(f"Loaded {key}: {len(result['samples']):,} posterior samples")

## Draw and optionally save the contour overlay

In [ ]:
selections = (
    [(key, "prior") for key in PRIORS]
    + [(key, "posterior") for key in POSTFITS]
)
figure = plot_distribution_overlay(
    results, selections, bins=BINS,
    n_reference_samples=N_PRIOR_SAMPLES, figsize=FIGSIZE,
)

if SAVE_FIGURE:
    output_dir = FIGURE_ROOT / SUITE / "comparison_overlays"
    output_dir.mkdir(parents=True, exist_ok=True)
    for extension in ("pdf", "png"):
        output_path = output_dir / f"{OUTPUT_STEM}.{extension}"
        figure.savefig(
            output_path, dpi=SAVE_DPI, bbox_inches="tight",
            pad_inches=.03, facecolor="white",
        )
        print("Saved:", output_path)
figure